# RQ1 follow-up: phân biệt cơ chế và tương tác

Notebook này **tự vá code trong runtime Colab** (không cần push). Dùng chung `runs/`, `data/` trên Drive với
`colab_photometric_rq1.ipynb`, nên 27 run cũ (N = 8) được tái sử dụng.

| Thí nghiệm | Câu hỏi | Huấn luyện? |
|---|---|---|
| **E0** | Độ nhạy theo ngưỡng τ ∈ {0.03, 0.05, 0.1, 0.15} | Không |
| **E2** | Floater sinh thêm có chủ yếu chỉ được 1–2 view huấn luyện nhìn thấy? (cơ chế b) | Không |
| **E1** | Nhiễu theo pixel, trung bình 0, cùng độ lớn: floater có tăng như nhiễu ISP? | 12 run |
| **E4** | Bù affine theo ảnh (12 tham số): floater có giảm về mức none? | 18 run |
| **E3** | Lưới N ∈ {4, 6, 8} × S: có tương tác? (P2) | 36 run |

E1, E3, E4 chỉ dùng scan63, scan83 (scan24 đang bất thường và chạy chậm). Chạy theo thứ tự: mục 1 → 2 → 3, rồi E0, E2 (vài phút),
rồi mục 5 (huấn luyện, có thể chạy lại nhiều lần: run đã xong được bỏ qua), cuối cùng mục 6.

## 1. Setup (giống notebook cũ)

In [ ]:
import os, subprocess, pathlib, shutil, sys
from google.colab import drive, userdata
drive.mount('/content/drive')
STORE = '/content/drive/MyDrive/faithfulrecon_store'
REPO_URL = 'github.com/Huyrua27/faithfulrecon.git'
BRANCH = 'feature/huy-dtu-multiscan'      # cùng code với 27 run đã có
REPO = '/content/faithfulrecon'
token = userdata.get('GH_TOKEN')
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, f'https://{token}@{REPO_URL}', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', f'https://{token}@{REPO_URL}', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '-q', '-f', '-B', BRANCH, 'FETCH_HEAD'], check=True)
subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'], check=True)
del token
for d in ('runs', 'data'):
    p = pathlib.Path(REPO, d)
    if p.is_symlink(): p.unlink()
    elif p.exists(): shutil.rmtree(p)
    p.symlink_to(f'{STORE}/{d}')
os.chdir(REPO); sys.path.insert(0, REPO); sys.path.insert(0, f'{REPO}/scripts')
!git log --oneline -1
!pip -q install -r requirements.txt

## 2. Vá code trong runtime (không commit)

1. `scripts/floater_metrics.py`: bản mở rộng (quét ngưỡng τ, số lượng, tổng độ mờ, tổng đóng góp).
2. `frecon/data/photometric.py`: thêm `mode: pixel` = nhiễu ISP nhưng **đảo dấu ngẫu nhiên theo từng pixel**:
   cùng độ lớn tại mỗi pixel, kỳ vọng bằng 0, không còn thành phần nhất quán theo ảnh.
3. `frecon/data/dtu.py`: tập kiểm định luôn lấy ngoài 8 camera FPS đầu tiên, để N = 4, 6, 8 dùng **cùng 16 view kiểm định**
   (tập huấn luyện lồng nhau; với N = 8 không đổi gì).
4. `frecon/train.py`: `train.appearance: affine` = mỗi ảnh huấn luyện có ma trận 3×3 + bias (12 tham số) áp lên ảnh kết xuất
   **chỉ khi tính loss**; checkpoint và đánh giá không dùng biến đổi này.

In [ ]:
%%writefile scripts/floater_metrics.py
"""Floater and geometry metrics of trained DTU base models (one row per run_dir x seed).

python scripts/floater_metrics.py --run_dirs runs/dtu_scan24_v8 runs/photo_strong_dtu_scan24_v8 \
    --seeds 0 1 2 --out results/photometric/floater_metrics.csv

Metrics (normalized units; d = distance from a Gaussian mean to the nearest reference point):
  off_003, off_01   fraction of Gaussians with d > 0.03 / d > 0.1
  front_01          fraction of Gaussians with d > 0.1 that lie IN FRONT of the reference surface
                    (median over training views of z_gauss - z_surface < -tol)
  wfloat_01         share of held-out blending weight carried by Gaussians with d > 0.1
                    (the floaters that actually render)
  acc, comp         mean distance Gaussian -> reference and reference -> Gaussian
                    (Gaussians with opacity > 0.05)
  psnr, dssim       held-out views, ORIGINAL (unperturbed) appearance
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import yaml

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from frecon.data.dtu import load_dtu_views, reference_points, split_views  # noqa: E402
from frecon.gaussians import nearest_dist  # noqa: E402
from frecon.measure import heldout_errors, render_views  # noqa: E402
from frecon.render import get_rasterizer  # noqa: E402
from frecon.train import TrainConfig, Trainer  # noqa: E402


def project(P, cam):
    Xc = P @ cam.R.T + cam.t
    z = Xc[:, 2]
    u = torch.floor(cam.fx * Xc[:, 0] / z.clamp_min(1e-6) + cam.cx).long()
    v = torch.floor(cam.fy * Xc[:, 1] / z.clamp_min(1e-6) + cam.cy).long()
    ok = (z > 1e-4) & (u >= 0) & (u < cam.W) & (v >= 0) & (v < cam.H)
    return z, u.clamp(0, cam.W - 1), v.clamp(0, cam.H - 1), ok


def zbuffer(ref, cam):
    z, u, v, ok = project(ref, cam)
    zb = torch.full((cam.H * cam.W,), float("inf"), device=ref.device)
    zb = zb.scatter_reduce(0, v[ok] * cam.W + u[ok], z[ok], "amin")
    return -F.max_pool2d(-zb.view(1, 1, cam.H, cam.W), 3, 1, 1)[0, 0]   # close small holes


@torch.no_grad()
def depth_side(means, views, tol):
    diffs = []
    for cam, mask, zb in views:
        z, u, v, ok = project(means, cam)
        zs = zb[v, u]
        ok &= mask[v, u] & torch.isfinite(zs)
        d = torch.full_like(z, float("nan"))
        d[ok] = z[ok] - zs[ok]
        diffs.append(d)
    return torch.nanmedian(torch.stack(diffs, 1), 1).values   # nan where no valid view


def load_eval_scene(cfg, dev):
    sc = cfg["scene"]
    assert sc.get("type") == "dtu", "floater_metrics.py supports DTU scenes only"
    cams, masks, scale_mat = load_dtu_views(sc["root"], sc.get("downscale", 4))
    tr, te = split_views(cams, sc["n_train_views"], sc["n_test_views"])
    ref = reference_points(sc.get("stl_ply"), scale_mat, cams, masks, sc.get("min_mask_views", 1),
                           sc.get("max_ref_points", 200_000), sc.get("ref_cache")).to(dev)
    views = []
    for i in tr:
        cam = cams[i].to(dev)
        views.append((cam, torch.from_numpy(masks[i]).to(dev), zbuffer(ref, cam)))
    return views, [cams[i].to(dev) for i in te], ref


def tau_tag(tau: float) -> str:
    """0.03 -> '003', 0.1 -> '01', 0.15 -> '015' (keeps the historical column names)."""
    return f"{tau:g}".replace("0.", "0", 1)


@torch.no_grad()
def metrics(model, views, test_cams, ref, rasterize, bg, tol, taus=(0.03, 0.05, 0.1, 0.15)):
    """W_float(tau) = sum_{k,p} sum_{i: d_i > tau} w_ik(p) / sum_{k,p} sum_i w_ik(p) over all pixels of
    the held-out views (torch renderer: exact blending weights, contributions < 1/255 dropped)."""
    means = model.means.detach()
    opac = model.opacities.detach().reshape(-1)
    d = nearest_dist(means, ref)
    med = depth_side(means, views, tol)
    contrib = torch.zeros(model.n, device=means.device)
    for cam in test_cams:
        contrib += model.render(rasterize, cam, bg, return_contrib=True)["contrib"]
    solid = opac > 0.05
    err = heldout_errors(render_views(model, test_cams, rasterize, bg), test_cams)
    out = {"n_gaussians": model.n}
    for tau in taus:
        far = d > tau
        t = tau_tag(tau)
        out[f"off_{t}"] = float(far.float().mean())
        out[f"wfloat_{t}"] = float(contrib[far].sum() / contrib.sum().clamp_min(1e-12))
        out[f"n_off_{t}"] = int(far.sum())                      # count, not a fraction
        out[f"opsum_off_{t}"] = float(opac[far].sum())          # total opacity of far Gaussians
        out[f"contrib_off_{t}"] = float(contrib[far].sum())     # absolute held-out weight
    off01 = d > 0.1
    out.update({
        "front_01": float(((med < -tol) & off01).sum() / off01.sum().clamp_min(1)),
        "contrib_total": float(contrib.sum()),
        "acc": float(d[solid].mean()),
        "comp": float(nearest_dist(ref, means[solid]).mean()),
        "psnr": err["psnr"], "dssim": err["dssim"],
    })
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--run_dirs", nargs="+", required=True)
    ap.add_argument("--seeds", type=int, nargs="+", default=[0, 1, 2])
    ap.add_argument("--tol", type=float, default=0.03)
    ap.add_argument("--taus", type=float, nargs="+", default=[0.03, 0.05, 0.1, 0.15],
                    help="distance thresholds (normalized units, scene ~ unit sphere) for the sweep")
    ap.add_argument("--out", default="results/photometric/floater_metrics.csv")
    args = ap.parse_args()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    rows, cache = [], {}
    for rd in map(Path, args.run_dirs):
        cfg = yaml.safe_load((rd / "config.yaml").read_text(encoding="utf-8"))
        rasterize = get_rasterizer(cfg.get("renderer", "torch"))
        bg = torch.ones(3, device=dev)
        key = cfg["scene"]["root"]
        if key not in cache:   # evaluation scene ignores `photometric`: original appearance
            cache = {key: load_eval_scene(cfg, dev)}
        views, test_cams, ref = cache[key]
        photo = cfg["scene"].get("photometric") or {}
        for seed in args.seeds:
            ckpt = rd / f"seed_{seed}" / "base.pt"
            if not ckpt.exists():
                print(f"skip {ckpt} (missing)")
                continue
            model = Trainer.load(ckpt, TrainConfig(**cfg["train"]), rasterize, bg, dev).model
            row = {"run": rd.name, "scan": Path(key).name, "seed": seed,
                   "exposure_stops": photo.get("exposure_stops", 0.0), "wb": photo.get("wb", 0.0),
                   "gamma": photo.get("gamma", 0.0)}
            row.update(metrics(model, views, test_cams, ref, rasterize, bg, args.tol, args.taus))
            rows.append(row)
            print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()}, flush=True)
            del model
            if dev == "cuda":
                torch.cuda.empty_cache()
    df = pd.DataFrame(rows)
    Path(args.out).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(args.out, index=False)
    print(f"-> {args.out}")


if __name__ == "__main__":
    main()

In [ ]:
from pathlib import Path

def patch(path, old, new, tag):
    s = Path(path).read_text(encoding='utf-8')
    if tag in s:
        print(f'{path}: already patched ({tag})'); return
    assert s.count(old) == 1, f'{path}: anchor not found for {tag}'
    Path(path).write_text(s.replace(old, new), encoding='utf-8'); print(f'{path}: patched ({tag})')

# (2) pixel-wise sign-flipped ISP noise
p = 'frecon/data/photometric.py'
s = Path(p).read_text(encoding='utf-8')
if 'PATCH_PIXEL_MODE' not in s:
    s += """

# PATCH_PIXEL_MODE (runtime patch from colab_rq1_followup.ipynb)
def perturb_views(cams, masks, cfg: dict):
    params = sample_params(len(cams), cfg.get("exposure_stops", 0.0), cfg.get("wb", 0.0),
                           cfg.get("gamma", 0.0), cfg.get("seed", 0))
    mode = cfg.get("mode", "isp")
    out = []
    for k, (cam, m, p) in enumerate(zip(cams, masks, params)):
        m = torch.as_tensor(np.asarray(m), device=cam.image.device).bool()
        img = apply(cam.image, m, p)
        if mode == "pixel":   # same per-pixel magnitude, random sign -> zero-mean, no per-view component
            g = torch.Generator().manual_seed(int(cfg.get("seed", 0)) * 1000 + k)
            sign = (torch.randint(0, 2, img.shape[:2], generator=g) * 2 - 1).to(img)[..., None]
            img = torch.where(m[..., None], (cam.image + sign * (img - cam.image)).clamp(0, 1), cam.image)
        out.append(replace(cam, image=img))
    return out, params
"""
    Path(p).write_text(s, encoding='utf-8'); print(p, ': patched (pixel mode)')

# (3) held-out views independent of N (nested FPS training sets)
patch('frecon/data/dtu.py',
      '    rest = [i for i in range(len(cams)) if i not in train_idx]',
      '    ref_idx = farthest_point_indices(centers, max(n_train, 8))   # PATCH_FIXED_TEST\n'
      '    rest = [i for i in range(len(cams)) if i not in ref_idx]',
      'PATCH_FIXED_TEST')

# (4) per-view affine appearance compensation (training loss only)
t = 'frecon/train.py'
patch(t, '    log_every: int = 500\n',
      '    log_every: int = 500\n'
      '    appearance: str = "none"        # PATCH_AFFINE: "affine" = per-view 3x3 + bias, loss only\n'
      '    lr_appearance: float = 1e-3\n', 'PATCH_AFFINE')
patch(t, '        self._reset_stats()\n\n    # ---- statistics used by densification',
      '        self._reset_stats()\n        self.app, self.app_opt = {}, None   # PATCH_APP_INIT\n\n'
      '    # ---- statistics used by densification', 'PATCH_APP_INIT')
patch(t, '        loss = photometric_loss(out["image"], cam.image, cfg.lambda_dssim)\n'
         '        self.opt.zero_grad(set_to_none=True)\n',
      '        loss = photometric_loss(self._appearance(out["image"], cam), cam.image, cfg.lambda_dssim)\n'
      '        self.opt.zero_grad(set_to_none=True)\n'
      '        if self.app_opt is not None:   # PATCH_APP_STEP\n'
      '            self.app_opt.zero_grad(set_to_none=True)\n', 'PATCH_APP_STEP')
patch(t, '        self.opt.step()\n        with torch.no_grad():  # keep quaternions',
      '        self.opt.step()\n        if self.app_opt is not None:   # PATCH_APP_OPT\n'
      '            self.app_opt.step()\n        with torch.no_grad():  # keep quaternions', 'PATCH_APP_OPT')
patch(t, '    def fit(self, cams,',
      '    def _appearance(self, img, cam):   # PATCH_APP_FN\n'
      '        if getattr(self.cfg, "appearance", "none") != "affine":\n'
      '            return img\n'
      '        if cam.name not in self.app:\n'
      '            A = torch.nn.Parameter(torch.eye(3, device=img.device))\n'
      '            b = torch.nn.Parameter(torch.zeros(3, device=img.device))\n'
      '            self.app[cam.name] = (A, b)\n'
      '            if self.app_opt is None:\n'
      '                self.app_opt = torch.optim.Adam([A, b], lr=self.cfg.lr_appearance)\n'
      '            else:\n'
      '                self.app_opt.add_param_group({"params": [A, b]})\n'
      '        A, b = self.app[cam.name]\n'
      '        return img @ A.T + b\n\n'
      '    def fit(self, cams,', 'PATCH_APP_FN')

# quick self-test of the patches (CPU, seconds)
import importlib, torch, numpy as np
import frecon.data.photometric as ph, frecon.train as tr
importlib.reload(ph); importlib.reload(tr)
from frecon.camera import Camera
cam = Camera(R=torch.eye(3), t=torch.tensor([0., 0., 2.]), fx=50., fy=50., cx=16., cy=16., W=32, H=32,
             image=torch.full((32, 32, 3), 0.5), name='c0')
m = np.ones((32, 32), bool)
(isp,), _ = ph.perturb_views([cam], [m], dict(exposure_stops=1.0, wb=0.15, gamma=0.2, seed=0))
(pix,), _ = ph.perturb_views([cam], [m], dict(exposure_stops=1.0, wb=0.15, gamma=0.2, seed=0, mode='pixel'))
di, dp = isp.image - cam.image, pix.image - cam.image
print('ISP   mean shift', di.mean().item(), '| |shift|', di.abs().mean().item())
print('pixel mean shift', dp.mean().item(), '| |shift|', dp.abs().mean().item())
assert abs(dp.mean().item()) < 0.2 * di.abs().mean().item()
assert tr.TrainConfig(appearance='affine').appearance == 'affine'
print('patches OK')

## 3. Cấu hình

In [ ]:
import yaml, json, time, torch, numpy as np, pandas as pd
SEEDS = [0, 1, 2]
LEVELS = {'none': None,
          'mild':   dict(exposure_stops=0.5, wb=0.05, gamma=0.10, seed=0),
          'strong': dict(exposure_stops=1.0, wb=0.15, gamma=0.20, seed=0)}
template = Path('configs/dtu_scan24_v8.yaml').read_text(encoding='utf-8')

def run_dir(s, N, level, tag=''):
    return f'runs/photo_{level}{tag}_dtu_scan{s}_v{N}'

def make_cfg(s, N, level, mode=None, appearance=None):
    tag = ('_pix' if mode == 'pixel' else '') + ('_aff' if appearance else '')
    base = yaml.safe_load(template.replace('scan24', f'scan{s}').replace('stl024', f'stl{s:03d}'))
    cfg = dict(base, out_dir=run_dir(s, N, level, tag), seeds=SEEDS)
    cfg['scene'] = dict(base['scene'], n_train_views=N)
    if LEVELS[level]:
        cfg['scene']['photometric'] = dict(LEVELS[level], **({'mode': mode} if mode else {}))
    if appearance:
        cfg['train'] = dict(base['train'], appearance=appearance)
    path = f"configs/{Path(cfg['out_dir']).name}.yaml"
    yaml.safe_dump(cfg, open(path, 'w', encoding='utf-8'), sort_keys=False)
    return path

EXPS = {   # name -> list of configs (N=8, ISP 'none/mild/strong' already exist from the first notebook)
    'E1_pixel':  [make_cfg(s, 8, l, mode='pixel') for s in (63, 83) for l in ('mild', 'strong')],
    'E4_affine': [make_cfg(s, 8, l, appearance='affine') for s in (63, 83) for l in LEVELS],
    'E3_views':  [make_cfg(s, N, l) for N in (6, 4) for s in (63, 83) for l in LEVELS],
}
for k, v in EXPS.items(): print(k, len(v) * len(SEEDS), 'runs')

## 4. E0 và E2: phân tích trên 27 checkpoint đã có (không huấn luyện)

**E2** (cơ chế b). Với mỗi Gaussian, đếm số view huấn luyện mà nó đóng góp ≥ 1 đơn vị trọng số trộn (`n_vis`).
Tách `W_float = W_float[n_vis ≤ 2] + W_float[n_vis ≥ 3]`. Dự đoán của cơ chế (b): mức tăng của W_float từ none tới strong
nằm chủ yếu ở thành phần `n_vis ≤ 2`.

In [ ]:
os.makedirs('results/rq1b', exist_ok=True)
runs8 = ' '.join(run_dir(s, 8, l) for s in (24, 63, 83) for l in LEVELS)
!python -W ignore scripts/floater_metrics.py --run_dirs {runs8} --seeds 0 1 2 --out results/rq1b/E0_thresholds.csv
e0 = pd.read_csv('results/rq1b/E0_thresholds.csv')
e0['level'] = e0.run.str.extract(r'photo_(\w+?)_dtu')[0]
cols = [c for c in e0.columns if c.startswith('wfloat_')] + [c for c in e0.columns if c.startswith('off_0')]
print(e0.groupby(['scan', 'level'])[cols].mean().reindex(list(LEVELS), level='level').round(3).to_string())

In [ ]:
from floater_metrics import load_eval_scene
from frecon.gaussians import nearest_dist
from frecon.render import get_rasterizer
from frecon.train import TrainConfig, Trainer
dev, EPS, TAU = 'cuda', 1.0, 0.1
rows = []
for s in (24, 63, 83):
    scene = None
    for l in LEVELS:
        rd = run_dir(s, 8, l)
        cfg = yaml.safe_load(open(f'{rd}/config.yaml', encoding='utf-8'))
        ras, bg = get_rasterizer(cfg.get('renderer', 'torch')), torch.ones(3, device=dev)
        scene = scene or load_eval_scene(cfg, dev)
        views, test_cams, ref = scene
        for seed in SEEDS:
            with torch.no_grad():
                m = Trainer.load(f'{rd}/seed_{seed}/base.pt', TrainConfig(**cfg['train']), ras, bg, dev).model
                far = nearest_dist(m.means.detach(), ref) > TAU
                nvis = torch.zeros(m.n, device=dev)
                for cam, _, _ in views:
                    nvis += m.render(ras, cam, bg, return_contrib=True)['contrib'] > EPS
                held = torch.zeros(m.n, device=dev)
                for cam in test_cams:
                    held += m.render(ras, cam, bg, return_contrib=True)['contrib']
                tot = held.sum()
                rows.append({'scan': f'scan{s}', 'level': l, 'seed': seed,
                             'W_float': float(held[far].sum() / tot),
                             'W_float_nvis_le2': float(held[far & (nvis <= 2)].sum() / tot),
                             'W_float_nvis_ge3': float(held[far & (nvis >= 3)].sum() / tot),
                             'mean_nvis_far': float(nvis[far].float().mean()),
                             'mean_nvis_surface': float(nvis[~far].float().mean()),
                             'n_far_nvis_le2': int((far & (nvis <= 2)).sum()), 'n_far': int(far.sum())})
            del m; torch.cuda.empty_cache()
        print(f'scan{s} {l} done', flush=True)
e2 = pd.DataFrame(rows); e2.to_csv('results/rq1b/E2_visibility.csv', index=False)
print(e2.groupby(['scan', 'level']).mean(numeric_only=True).drop(columns='seed')
        .reindex(list(LEVELS), level='level').round(3).to_string())
base = e2[e2.level == 'none'].set_index(['scan', 'seed'])
for l in ('mild', 'strong'):
    d = e2[e2.level == l].set_index(['scan', 'seed'])[['W_float', 'W_float_nvis_le2', 'W_float_nvis_ge3']] \
        - base[['W_float', 'W_float_nvis_le2', 'W_float_nvis_ge3']]
    share = (d['W_float_nvis_le2'] / d['W_float']).replace([np.inf, -np.inf], np.nan)
    print(f'\n{l}: ΔW_float {d.W_float.mean():+.3f} = Δ[n_vis<=2] {d.W_float_nvis_le2.mean():+.3f} '
          f'+ Δ[n_vis>=3] {d.W_float_nvis_ge3.mean():+.3f}; share from n_vis<=2: median {share.median():.2f}')

## 5. Huấn luyện E1, E4, E3 (chạy lại được; run đã xong sẽ bỏ qua)

Thứ tự ưu tiên: E1 và E4 (trả lời câu hỏi cơ chế, N = 8) trước, E3 sau. Mỗi cấu hình chạy 3 seed song song.
Ước lượng trên L4: khoảng 20–35 phút cho mỗi cấu hình scan63/83 với N = 8, ít hơn khi N nhỏ.

In [ ]:
RUN = ['E1_pixel', 'E4_affine', 'E3_views']    # bỏ bớt tên để chạy riêng một thí nghiệm
def done(path):
    od = yaml.safe_load(open(path, encoding='utf-8'))['out_dir']
    return all(os.path.exists(f'{od}/seed_{k}/base.pt') for k in SEEDS)
for e in RUN:
    for path in EXPS[e]:
        if done(path):
            print('skip', path); continue
        t0 = time.time()
        code_ = subprocess.call([sys.executable, 'scripts/run_parallel.py', '--script', 'scripts/phase0.py',
                                 '--config', path, '--workers', str(len(SEEDS)), '--base_only'])
        print(f'{e}: {path} -> exit {code_} ({(time.time() - t0) / 60:.1f} min)', flush=True)

In [ ]:
# tiến độ: số checkpoint đã có / tổng
for e, paths in EXPS.items():
    n = sum(os.path.exists(f"{yaml.safe_load(open(p))['out_dir']}/seed_{k}/base.pt") for p in paths for k in SEEDS)
    print(f'{e}: {n}/{len(paths) * len(SEEDS)}')

## 6. Phân tích E1, E4, E3

In [ ]:
def metrics_for(paths, out):
    dirs = [yaml.safe_load(open(p))['out_dir'] for p in paths]
    subprocess.run([sys.executable, '-W', 'ignore', 'scripts/floater_metrics.py', '--run_dirs', *dirs,
                    '--seeds', '0', '1', '2', '--out', out], check=True)
    return pd.read_csv(out)

ref8 = e0[e0.scan.isin(['scan63', 'scan83'])].assign(N=8, variant='isp')
parts = [ref8]
if all(os.path.exists(f"{yaml.safe_load(open(p))['out_dir']}/seed_0/base.pt") for p in EXPS['E1_pixel']):
    parts.append(metrics_for(EXPS['E1_pixel'], 'results/rq1b/E1_pixel.csv').assign(N=8, variant='pixel'))
if all(os.path.exists(f"{yaml.safe_load(open(p))['out_dir']}/seed_0/base.pt") for p in EXPS['E4_affine']):
    parts.append(metrics_for(EXPS['E4_affine'], 'results/rq1b/E4_affine.csv').assign(N=8, variant='affine'))
if all(os.path.exists(f"{yaml.safe_load(open(p))['out_dir']}/seed_0/base.pt") for p in EXPS['E3_views']):
    e3 = metrics_for(EXPS['E3_views'], 'results/rq1b/E3_views.csv')
    e3['N'] = e3.run.str.extract(r'_v(\d+)$')[0].astype(int)
    parts.append(e3.assign(variant='isp'))
allm = pd.concat(parts, ignore_index=True)
allm['level'] = allm.run.str.extract(r'photo_(none|mild|strong)')[0]
allm.to_csv('results/rq1b/all_metrics.csv', index=False)
M = ['wfloat_01', 'wfloat_005', 'off_01', 'acc', 'comp', 'n_gaussians']
print(allm.groupby(['variant', 'N', 'scan', 'level'])[M].mean().round(3).to_string())

In [ ]:
# E1: ISP (có hệ thống) vs pixel (không hệ thống), cùng độ lớn. Δ so với none cùng scan, seed.
def delta(df, by):
    b = df[df.level == 'none'].set_index(['scan', 'seed'])['wfloat_01']
    out = []
    for key, g in df[df.level != 'none'].groupby(by):
        d = g.set_index(['scan', 'seed'])['wfloat_01'] - b
        out.append(dict(zip(by, key if isinstance(key, tuple) else (key,)), dW=d.mean(), up=f'{(d > 0).sum()}/{d.notna().sum()}'))
    return pd.DataFrame(out)
none8 = allm[(allm.N == 8) & (allm.variant == 'isp') & (allm.level == 'none')]
for v in ('pixel',):
    sub = pd.concat([none8, allm[(allm.N == 8) & (allm.variant.isin(['isp', v])) & (allm.level != 'none')]])
    print('E1:\n', delta(sub, ['variant', 'level']).round(3).to_string(index=False))
# E4: affine vs không affine, cùng mức nhiễu
if (allm.variant == 'affine').any():
    t = allm[allm.N == 8].pivot_table(index=['scan', 'level', 'seed'], columns='variant', values='wfloat_01')
    t['affine - isp'] = t['affine'] - t['isp']
    print('\nE4 (W_float, N=8):\n', t.groupby(level=['scan', 'level']).mean().round(3).to_string())

In [ ]:
# E3: tương tác N x S. ΔS(N) = W(N, S) - W(N, none) theo cặp scan, seed; và hồi quy logit có hiệu ứng scan.
e = allm[allm.variant == 'isp'].copy()
if e.N.nunique() > 1:
    rows = []
    for (N, s), g in e.groupby(['N', 'scan']):
        b = g[g.level == 'none'].set_index('seed')['wfloat_01']
        for l in ('mild', 'strong'):
            d = g[g.level == l].set_index('seed')['wfloat_01'] - b
            rows.append({'N': N, 'scan': s, 'level': l, 'dW': d.mean(), 'sd': d.std()})
    print(pd.DataFrame(rows).pivot_table(index=['scan', 'level'], columns='N', values='dW').round(3).to_string())
    S = e.level.map({'none': 0., 'mild': 0.5, 'strong': 1.})
    y = np.log(e.wfloat_01 / (1 - e.wfloat_01))
    X = np.column_stack([np.ones(len(e)), e.N, S, e.N * S] + [(e.scan == sc).astype(float) for sc in sorted(e.scan.unique())[1:]])
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    r = y - X @ beta; se = np.sqrt(np.diag(np.linalg.inv(X.T @ X)) * (r @ r) / (len(y) - X.shape[1]))
    for nm, b_, s_ in zip(['b0', 'N', 'S', 'N*S'], beta, se):
        print(f'{nm:4s} {b_:+.4f} ± {s_:.4f}')
    print('Tương tác: hệ số N*S âm nghĩa là tác động của S lớn hơn khi N nhỏ (phù hợp P2).')

In [ ]:
!mkdir -p {STORE}/results/rq1b && cp results/rq1b/* {STORE}/results/rq1b/ && ls {STORE}/results/rq1b